# Inventory Pickings: Bronze to Silver

This notebook uses the same production transformations and callback as the executable job.


In [ ]:
from pathlib import Path
import os, sys, uuid
_cwd = Path.cwd().resolve()
_candidates = [_cwd, *_cwd.parents]
PROJECT_ROOT = next((p for p in _candidates if (p / "pipelines").is_dir()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = next((p / "repo" / "product_analysis" for p in _candidates if (p / "repo" / "product_analysis" / "pipelines").is_dir()), None)
if PROJECT_ROOT is None: raise RuntimeError("Could not locate product_analysis project root")
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))


In [ ]:
from pipelines.multi_dataset_silver import DatasetRuntimeConfig, load_dataset_schema, create_dataset_tables, build_bronze_stream, build_dataset_batch_processor
from pipelines.inventory_pickings_silver import SPEC
from pipelines.bronze_to_silver_common import configure_logging
from pipelines.landing_to_bronze_common import start_landing_spark
logger = configure_logging(os.environ.get("LOG_LEVEL", "INFO"))
config = DatasetRuntimeConfig(dataset_name="inventory_pickings", bronze_path=os.environ.get("BRONZE_PATH", "hdfs://localhost:9000/capstone/warehouse/bronze/inventory_pickings"), checkpoint_path=os.environ.get("CHECKPOINT_PATH", "hdfs://localhost:9000/capstone/checkpoints/bronze_to_silver/inventory_pickings"), silver_table=os.environ.get("SILVER_TABLE", "datalake.silver.inventory_pickings"), rejected_table=os.environ.get("REJECTED_TABLE", "datalake.quarantine.inventory_pickings_rejected"), audit_table=os.environ.get("AUDIT_TABLE", "datalake.operations.silver_ingestion_audit"), exploration_table=os.environ.get("EXPLORATION_TABLE", "datalake.operations.explorations"), expectation_table=os.environ.get("EXPECTATION_TABLE", "datalake.operations.expectation_profiles"), schema_file=str(PROJECT_ROOT / "schemas" / "inventory_pickings_schema.json"), pipeline_run_id=os.environ.get("PIPELINE_RUN_ID", f"inventory_pickings-{uuid.uuid4()}"), max_files_per_trigger=int(os.environ.get("MAX_FILES_PER_TRIGGER", "5")))


In [ ]:
spark = start_landing_spark("inventory_pickings_bronze_to_silver", config.catalog)


In [ ]:
business_schema = load_dataset_schema(config.schema_file)
create_dataset_tables(spark, spec=SPEC, config=config)


In [ ]:
bronze_stream = build_bronze_stream(spark, config, business_schema)
batch_processor = build_dataset_batch_processor(spec=SPEC, config=config, logger=logger)
query = (bronze_stream.writeStream.queryName(f"{config.dataset_name}_bronze_to_silver").foreachBatch(batch_processor).option("checkpointLocation", config.checkpoint_path).trigger(availableNow=True).start())
query.awaitTermination()
if query.exception() is not None: raise RuntimeError(query.exception())


In [ ]:
spark.catalog.refreshTable(config.silver_table)
silver = spark.table(config.silver_table)
print("Silver count:", silver.count())
print("Distinct key count:", silver.select(SPEC.key_output).distinct().count())
print("Current run audit:")
spark.table(config.audit_table).where(f"pipeline_run_id = '{config.pipeline_run_id}'").show(truncate=False)


In [ ]:
RUN_CHECKPOINT_RERUN = False
if RUN_CHECKPOINT_RERUN:
    rerun = (bronze_stream.writeStream.queryName(f"{config.dataset_name}_checkpoint_validation").foreachBatch(batch_processor).option("checkpointLocation", config.checkpoint_path).trigger(availableNow=True).start())
    rerun.awaitTermination()
    if rerun.exception() is not None: raise RuntimeError(rerun.exception())
